## Refinamiento de hiperparametros con Optuna

### Imports

In [ ]:
import optuna
from ultralytics import YOLO
import torch
from pathlib import Path
import sys

root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))
    
from config import YAML_PATH, CUSTOM_MODEL_WEIGHTS_PATH

In [ ]:
ITERACIONES_OPTUNA = 100

### Funcion objetivo

In [ ]:
def objective(trial, model):
    #Hiperparámetros
    imgsz = trial.suggest_categorical("imgsz", [640, 960, 1280])
    
    #Ajustamos el batch según la resolución para evitar OOM (Out of Memory)
    if imgsz == 1280:
        batch = trial.suggest_categorical("batch", [4, 8])
    elif imgsz == 960:
        batch = trial.suggest_categorical("batch", [8, 16])
    else:
        batch = trial.suggest_categorical("batch", [16, 32])
        
    lr0 = trial.suggest_float("lr0", 1e-4, 5e-3, log=True)
    lrf = trial.suggest_float("lrf", 0.01, 0.1)
    momentum = trial.suggest_float("momentum", 0.7, 0.98)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    
    #Hiperparámetros de aumento 
    mosaic = trial.suggest_float("mosaic", 0.3, 1.0)
    degrees = trial.suggest_float("degrees", 0.0, 10.0)
    scale = trial.suggest_float("scale", 0.3, 0.6)
    
    model = YOLO(CUSTOM_MODEL_WEIGHTS_PATH)

    #Entrenar el modelo con los hiperparámetros sugeridos
    results = model.train(
        data=YAML_PATH,
        epochs=40,
        imgsz=imgsz,
        lr0=lr0,
        lrf=lrf,
        momentum=momentum,
        weight_decay=weight_decay,
        batch=batch,
        mosaic=mosaic,
        degrees=degrees,
        scale=scale,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        seed=42,
        project="optuna_alpr",
        name=f"trial_{trial.number}",
        exist_ok=True,
        cache=True,
        plots=False
    )

    #Métrica objetivo
    #mAP 50-95 en el set de validación
    map50_95 = results.results_dict['metrics/m_ap_50-95']
    
    torch.cuda.empty_cache()
    
    return map50_95

### Estudio

In [ ]:
sampler = optuna.samplers.TPESampler(
    n_startup_trials=10,
    multivariate=True,
    seed=42
)

study = optuna.create_study(direction="maximize", sampler=sampler, pruner=optuna.pruners.MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=ITERACIONES_OPTUNA, show_progress_bar=True)

print(study.best_params)